# 🚗 AI-Based Vehicle Fuel Consumption Prediction System
## Mechanical & Automotive Engineering ML Project

### 📌 Project Overview & Problem Statement
In automotive design and mechanical engineering, predicting vehicle fuel consumption is essential for optimizing engine efficiency, reducing greenhouse gas emissions, and complying with fuel economy standards.

Instead of relying solely on empirical thermodynamic formulas, this notebook demonstrates how to train **Machine Learning Regression Models** on historical vehicle parameters (`Engine Size`, `Horsepower`, `Vehicle Weight`, `Cylinders`, `Acceleration`, `Fuel Type`, `Transmission`) to accurately predict fuel consumption (**L/100 km**).

---

## Step 1: Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Graphics styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
print("Libraries loaded successfully!")

## Step 2: Load and Inspect Dataset

In [ ]:
# Load dataset
data_path = os.path.join("..", "data", "vehicle_fuel_dataset.csv")
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape}")
display(df.head(10))

In [ ]:
# Summary Statistics & Data Types
print("--- Data Types & Missing Values ---")
df.info()

print("\n--- Numerical Feature Statistics ---")
display(df.describe())

## Step 3: Exploratory Data Analysis (EDA)
Let's investigate how physical features correlate with Fuel Consumption.

In [ ]:
# Feature Correlation Heatmap
num_cols = ['Engine Size', 'Cylinders', 'Horsepower', 'Vehicle Weight', 'Acceleration', 'Model Year', 'Fuel Consumption']
plt.figure(figsize=(10, 7))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="YlGnBu", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Vehicle Features vs Fuel Consumption", fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Scatter Plots: Vehicle Weight & Engine Size vs Fuel Consumption
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.scatterplot(data=df, x="Vehicle Weight", y="Fuel Consumption", hue="Fuel Type", palette="tab10", ax=axes[0])
axes[0].set_title("Vehicle Weight (kg) vs Fuel Consumption (L/100 km)", fontweight='bold')

sns.scatterplot(data=df, x="Engine Size", y="Fuel Consumption", hue="Fuel Type", palette="tab10", ax=axes[1])
axes[1].set_title("Engine Size (L) vs Fuel Consumption (L/100 km)", fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Categorical Analysis: Impact of Fuel Type and Transmission
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=df, x="Fuel Type", y="Fuel Consumption", palette="Set2", ax=axes[0])
axes[0].set_title("Fuel Consumption Distribution by Fuel Type", fontweight='bold')

sns.boxplot(data=df, x="Transmission", y="Fuel Consumption", palette="Set3", ax=axes[1])
axes[1].set_title("Fuel Consumption Distribution by Transmission", fontweight='bold')

plt.tight_layout()
plt.show()

## Step 4: Data Preprocessing & Train/Test Split

In [ ]:
# Define Features (X) and Target (y)
num_features = ['Engine Size', 'Cylinders', 'Horsepower', 'Vehicle Weight', 'Acceleration', 'Model Year']
cat_features = ['Fuel Type', 'Transmission']
target = 'Fuel Consumption'

X = df[num_features + cat_features]
y = df[target]

# ColumnTransformer for scaling numeric features and one-hot encoding categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
    ]
)

# Train/Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"X_train shape: {X_train_proc.shape}")
print(f"X_test shape: {X_test_proc.shape}")

## Step 5: Model Training & Comparison
We will train and evaluate three regression models:
1. **Linear Regression** (Parametric linear baseline)
2. **Decision Tree Regressor** (Non-linear rule-based tree)
3. **Random Forest Regressor** (Ensemble of decision trees)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
}

results = {}

for name, model in models.items():
    # Train
    model.fit(X_train_proc, y_train)
    preds = model.predict(X_test_proc)
    
    # Evaluate
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    
    results[name] = {"MAE": mae, "RMSE": rmse, "R2 Score": r2}

# Display Comparison Table
metrics_df = pd.DataFrame(results).T
display(metrics_df)

In [ ]:
# Plot Metric Comparisons
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics_df[['MAE', 'RMSE']].plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'], rot=0)
axes[0].set_title("MAE & RMSE Comparison (Lower is Better)", fontweight='bold')
axes[0].set_ylabel("L/100 km")

metrics_df['R2 Score'].plot(kind='bar', ax=axes[1], color='#2ecc71', rot=0)
axes[1].set_title("R² Score Comparison (Higher is Better)", fontweight='bold')
axes[1].set_ylim(0.8, 1.0)

plt.tight_layout()
plt.show()

## Step 6: Feature Importance Analysis (Random Forest)

In [ ]:
# Feature Importance Breakdown
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_cols = list(cat_encoder.get_feature_names_out(cat_features))
feature_names = num_features + encoded_cat_cols

rf_model = models["Random Forest"]
importances = pd.Series(rf_model.feature_importances_, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importances.plot(kind='barh', color='#9b59b6')
plt.title("Random Forest Feature Importances for Fuel Consumption", fontweight='bold', fontsize=14)
plt.xlabel("Relative Importance")
plt.show()

## Step 7: Sample Prediction & Saving Trained Models

In [ ]:
# Test prediction on a sample sedan
sample_car = pd.DataFrame([{
    'Engine Size': 2.0,
    'Cylinders': 4,
    'Horsepower': 160,
    'Vehicle Weight': 1420,
    'Acceleration': 8.8,
    'Model Year': 2022,
    'Fuel Type': 'Petrol',
    'Transmission': 'Automatic'
}])

sample_proc = preprocessor.transform(sample_car)
pred_fc = rf_model.predict(sample_proc)[0]
pred_mpg = 235.215 / pred_fc

print(f"Predicted Fuel Consumption for Sample Vehicle:")
print(f"- {round(pred_fc, 2)} L/100 km ({round(pred_mpg, 1)} MPG)")

In [ ]:
# Export artifacts using Joblib
output_dir = os.path.join("..", "models", "saved_models")
os.makedirs(output_dir, exist_ok=True)

joblib.dump(preprocessor, os.path.join(output_dir, "preprocessor.joblib"))
for name, model in models.items():
    fname = f"regressor_{name.lower().replace(' ', '_')}.joblib"
    joblib.dump(model, os.path.join(output_dir, fname))
    
print(f"All artifacts saved to {output_dir}")